# Handwired native Vulkan WGSL on NVIDIA Tesla T4
Runs the **actual Handwired Circuit-generated pressure WGSL** via wgpu-native on physical NVIDIA GPUs. This is not browser WebGPU; the result must be labeled native hardware WGSL only.


In [ ]:
import pathlib,subprocess,json,time,os,traceback
base=pathlib.Path('/kaggle/working')
report={'ok':False,'stages':[],'webGpuBrowser':False}
def run(label,cmd,strict=True):
 print('##',label,flush=True)
 p=subprocess.run(cmd,shell=True,executable='/bin/bash',text=True,
   stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=1300)
 print(p.stdout[-5500:],flush=True)
 report['stages'].append({'name':label,'exitCode':p.returncode,
  'outputTail':p.stdout[-1600:]})
 (base/'native-vulkan-job.json').write_text(json.dumps(report,indent=2))
 if strict and p.returncode!=0:raise RuntimeError(label+' failed')
 return p
try:
 run('Identity of CUDA resources','nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
 run('NVIDIA Vulkan ICD','apt-get update -qq && apt-get install -y -qq libnvidia-gl-580 libvulkan1 vulkan-tools',True)
 run('Vulkan physical adapters','vulkaninfo --summary | grep -E "deviceName|deviceType|driverName" | head -16')
 run('Install wgpu-native and Bun','python -m pip -q install wgpu==0.32.0 && curl -fsSL https://bun.sh/install | bash')
 run('Clone actual feature branch','git clone --depth 1 --branch feat/frontier-02-cataclysm https://github.com/Chrisbryan17/handwired.git /kaggle/temp/handwired')
 run('Bun dependencies','cd /kaggle/temp/handwired && /root/.bun/bin/bun install --frozen-lockfile')
 run('Export real Circuit WGSL + independent reference','cd /kaggle/temp/handwired && /root/.bun/bin/bun scripts/export-frontier-pressure.ts /kaggle/working')
 run('Hardware native WGSL evaluation','python /kaggle/temp/handwired/ops/kaggle/native-wgpu/verify_pressure.py /kaggle/working')
 r=json.loads((base/'handwired-native-wgsl-report.json').read_text())
 report['ok']=r.get('ok',False)
 report['nativeGpuEvidence']=r
except Exception as e:
 report['error']=str(e)
 traceback.print_exc()
finally:
 report['finished']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())
 (base/'native-vulkan-job.json').write_text(json.dumps(report,indent=2))
print('NATIVE_GPU_WGSL_VERIFIED=',report['ok'],flush=True)
